In [62]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min
from sklearn.metrics import silhouette_score
df=pd.read_csv('HR_data_2.csv',index_col=0)

In [63]:
features=['HR_TD_Mean','HR_TD_std','TEMP_TD_Mean','TEMP_TD_std','EDA_TD_P_Mean','EDA_TD_T_Mean']
df_clean=df.dropna(subset=features+['Phase', 'Frustrated']).copy()
print("clean")

clean


In [64]:
train=df_clean[df_clean['Phase']=='phase1'][features]
scaler=StandardScaler()
X_train=scaler.fit_transform(train)
model=OneClassSVM(kernel='rbf', nu=0.05)
model.fit(X_train)
print("trained")

trained


vi skal lige vælge rigtig kernal og måske andet, RBF bliver brugt i slides


In [ ]:
# fit all three detectors on the pooled resting baseline
svm=OneClassSVM(kernel='rbf',nu=0.05).fit(X_train)
gmm=GaussianMixture(n_components=2,random_state=42).fit(X_train)
# anomaly threshold: bottom 5% of resting log-likelihoods
gmm_t=np.percentile(gmm.score_samples(X_train),5)
km=KMeans(n_clusters=2,random_state=42,n_init=10).fit(X_train)
_,tr_d=pairwise_distances_argmin_min(X_train,km.cluster_centers_)
# anything farther than the 95th-percentile training distance counts as anomalous
km_t=np.percentile(tr_d,95)

# evaluate each model on puzzle (phase2) and recovery (phase3)
for p in ['phase2','phase3']:
    test=df_clean[df_clean['Phase']==p].copy()
    X_test=scaler.transform(test[features])
    test['svm_pred']=svm.predict(X_test)
    test['gmm_pred']=np.where(gmm.score_samples(X_test)<gmm_t,-1,1)
    _,te_d=pairwise_distances_argmin_min(X_test,km.cluster_centers_)
    test['km_pred']=np.where(te_d>km_t,-1,1)
    for m in ['svm','gmm','km']:
        f=(test[f'{m}_pred']==-1).sum()
        print(f"{p} {m} anomaly %:{f}/{len(test)} ({100*f/len(test):.1f}%)")

phase2 svm anomaly %:19/104 (18.3%)
phase2 gmm anomaly %:17/104 (16.3%)
phase2 km anomaly %:5/104 (4.8%)
phase3 svm anomaly %:30/104 (28.8%)
phase3 gmm anomaly %:19/104 (18.3%)
phase3 km anomaly %:8/104 (7.7%)


In [ ]:
subjects=df_clean['Individual'].unique()
results=[]
for p in ['phase2','phase3']:
    for s in subjects:
        # train on everyone except the held-out subject (LOSO scheme)
        train=df_clean[(df_clean['Individual']!=s)&(df_clean['Phase']=='phase1')][features]
        test=df_clean[(df_clean['Individual']==s)&(df_clean['Phase']==p)][features]
        if len(train)<5 or len(test)==0:
            continue
        sc=StandardScaler().fit(train)
        Xt=sc.transform(train)
        Xs=sc.transform(test)

        # OC-SVM anomaly count for this subject
        svm=OneClassSVM(kernel='rbf',nu=0.05).fit(Xt)
        fs=(svm.predict(Xs)==-1).sum()

        # pick GMM component count via silhouette score (k=2..5)
        bkg=2
        msg=-1
        for k in range(2,6):
            gtmp=GaussianMixture(n_components=k,random_state=42).fit(Xt)
            l=gtmp.predict(Xt)
            if len(np.unique(l))>1:
                sil=silhouette_score(Xt,l)
                if sil>msg:
                    msg=sil
                    bkg=k
        gmm=GaussianMixture(n_components=bkg,random_state=42).fit(Xt)
        gt=np.percentile(gmm.score_samples(Xt),5)
        fg=(np.where(gmm.score_samples(Xs)<gt,-1,1)==-1).sum()

        # pick KMeans k the same way
        bkk=2
        msk=-1
        for k in range(2,6):
            ktmp=KMeans(n_clusters=k,random_state=42,n_init=10).fit(Xt)
            sil=silhouette_score(Xt,ktmp.labels_)
            if sil>msk:
                msk=sil
                bkk=k
        km=KMeans(n_clusters=bkk,random_state=42,n_init=10).fit(Xt)
        _,tr_d=pairwise_distances_argmin_min(Xt,km.cluster_centers_)
        kt=np.percentile(tr_d,95)
        _,te_d=pairwise_distances_argmin_min(Xs,km.cluster_centers_)
        fk=(te_d>kt).sum()

        results.append({'phase':p,'subject':s,'n_test':len(test),'svm_pct':100*fs/len(test),'gmm_pct':100*fg/len(test),'km_pct':100*fk/len(test)})

loso=pd.DataFrame(results)
print(loso.to_string(index=False))
print("\nmean anomaly rates:")
for p in ['phase2','phase3']:
    for m in ['svm','gmm','km']:
        print(f"{p} {m} anomaly %: {loso[loso['phase']==p][f'{m}_pct'].mean():.1f}%")

 phase  subject  n_test  svm_pct  gmm_pct  km_pct
phase2        1       4      0.0      0.0     0.0
phase2        2       4      0.0      0.0     0.0
phase2        3       4      0.0      0.0     0.0
phase2        4       4      0.0     25.0     0.0
phase2        5       4     25.0      0.0     0.0
phase2        6       4     75.0      0.0     0.0
phase2        7       4      0.0      0.0     0.0
phase2        8       4    100.0      0.0     0.0
phase2        9       4     50.0      0.0     0.0
phase2       10       4     25.0      0.0     0.0
phase2       11       4      0.0      0.0     0.0
phase2       12       4      0.0     25.0     0.0
phase2       13       4      0.0      0.0     0.0
phase2       14       4      0.0      0.0     0.0
phase2       15       4      0.0     25.0     0.0
phase2       16       4     25.0      0.0     0.0
phase2       17       4     25.0     25.0     0.0
phase2       18       4     50.0     75.0    25.0
phase2       19       4    100.0    100.0   100.0
